# Final Project

Task: The telecom operator Interconnect would like to be able to forecast their churn of clients.

**Assessment criteria:**

AUC-ROC < 0.75 — 0 SP

0.75 ≤ AUC-ROC < 0.81 — 4 SP

0.81 ≤ AUC-ROC < 0.85 — 4.5 SP

0.85 ≤ AUC-ROC < 0.87 — 5 SP

0.87 ≤ AUC-ROC < 0.88 — 5.5 SP

AUC-ROC ≥ 0.88 — 6 SP

## Work Plan Summary

1. Inspect datasets and understand feature types
2. Merge all tables using customerID
3. Create the target variable from EndDate
4. Clean and preprocess the data:
    * Convert data types where necessary
    * Encode categorical variables
5. Split the dataset into training and testing sets
6. Train and evaluate several classification models:
    * Logistic Regression (baseline)
    * Random Forest
    * LightGBM
7. Evaluate models using ROC-AUC (primary metric)
8. Select the best-performing model and provide conclusions

In [1]:
import pandas as pd

#Get Datasets
contract = pd.read_csv('/datasets/final_provider/contract.csv')
personal = pd.read_csv('/datasets/final_provider/personal.csv')
internet = pd.read_csv('/datasets/final_provider/internet.csv')
phone= pd.read_csv('/datasets/final_provider/phone.csv')

In [2]:
# Inspect and understand features
contract.info()
contract.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB


,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65


In [3]:
# Inspect and understand features
personal.info()
personal.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     7043 non-null   object
 1   gender         7043 non-null   object
 2   SeniorCitizen  7043 non-null   int64 
 3   Partner        7043 non-null   object
 4   Dependents     7043 non-null   object
dtypes: int64(1), object(4)
memory usage: 275.2+ KB


,customerID,gender,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No


In [4]:
# Inspect and understand features
internet.info()
internet.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5517 entries, 0 to 5516
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        5517 non-null   object
 1   InternetService   5517 non-null   object
 2   OnlineSecurity    5517 non-null   object
 3   OnlineBackup      5517 non-null   object
 4   DeviceProtection  5517 non-null   object
 5   TechSupport       5517 non-null   object
 6   StreamingTV       5517 non-null   object
 7   StreamingMovies   5517 non-null   object
dtypes: object(8)
memory usage: 344.9+ KB


,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No


In [5]:
# Inspect and understand features
phone.info()
phone.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6361 entries, 0 to 6360
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     6361 non-null   object
 1   MultipleLines  6361 non-null   object
dtypes: object(2)
memory usage: 99.5+ KB


,customerID,MultipleLines
0,5575-GNVDE,No
1,3668-QPYBK,No
2,9237-HQITU,No
3,9305-CDSKC,Yes
4,1452-KIOVK,Yes


In [6]:
#Merge tables
df = contract.merge(personal, on='customerID', how='left') \
             .merge(internet, on='customerID', how='left') \
             .merge(phone, on='customerID', how='left')

In [7]:
#General Inforamtion
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
 8   gender            7043 non-null   object 
 9   SeniorCitizen     7043 non-null   int64  
 10  Partner           7043 non-null   object 
 11  Dependents        7043 non-null   object 
 12  InternetService   5517 non-null   object 
 13  OnlineSecurity    5517 non-null   object 
 14  OnlineBackup      5517 non-null   object 
 15  DeviceProtection  5517 non-null   object 
 16  TechSupport       5517 non-null   object 


,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,gender,SeniorCitizen,Partner,Dependents,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,MultipleLines
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,Yes,No,DSL,No,Yes,No,No,No,No,NaN
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5,Male,0,No,No,DSL,Yes,No,Yes,No,No,No,No
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,No,No,DSL,Yes,Yes,No,No,No,No,No
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,No,No,DSL,Yes,No,Yes,Yes,No,No,NaN
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,No,No,Fiber optic,No,No,No,No,No,No,No


In [8]:
#Create the target
df['target'] = (df['EndDate'] == 'No').astype(int)

Created column for the target:This line selects the EndDate column from the dataframe, and creates a boolean (true/false) with No being: customer is still active. And Date being:customer left (churned).

Then is turned in to an integrer as ML requires numbers, in this case binary:1 = customer stayed and 0 = customer left

In [9]:
#Create a numeric value for "BeginDate"
df['BeginDate'] = pd.to_datetime(df['BeginDate'], errors='coerce')

report_date = pd.to_datetime('2020-02-01') 
df['days_as_customer'] = (report_date - df['BeginDate']).dt.days

Some models like Logistic regression Need numbers with interpretable meaning and do not process date type data. So I decided to turn it into a meaningful number as the tenure as a customer

In [10]:
#Drop Columns
df = df.drop(columns=['EndDate', 'customerID', 'BeginDate'])

Drop 'EndDate' as it is the target

Drop 'customerID' and 'BeginDate' as they won't help the Machines 

In [11]:
df.head()

,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,gender,SeniorCitizen,Partner,Dependents,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,MultipleLines,target,days_as_customer
0,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,Yes,No,DSL,No,Yes,No,No,No,No,NaN,1,31
1,One year,No,Mailed check,56.95,1889.5,Male,0,No,No,DSL,Yes,No,Yes,No,No,No,No,1,1036
2,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,No,No,DSL,Yes,Yes,No,No,No,No,No,0,123
3,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,No,No,DSL,Yes,No,Yes,Yes,No,No,NaN,1,1371
4,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,No,No,Fiber optic,No,No,No,No,No,No,No,0,153


In [12]:
#Clean Data

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

df.isna().sum()

Type                   0
PaperlessBilling       0
PaymentMethod          0
MonthlyCharges         0
TotalCharges          11
gender                 0
SeniorCitizen          0
Partner                0
Dependents             0
InternetService     1526
OnlineSecurity      1526
OnlineBackup        1526
DeviceProtection    1526
TechSupport         1526
StreamingTV         1526
StreamingMovies     1526
MultipleLines        682
target                 0
days_as_customer       0
dtype: int64

In [13]:
#Fill NaN in former internet and phone dataframes

cols = [
    'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'MultipleLines'
]

df[cols] = df[cols].fillna("No")

NaN values from the former internet and phone df are able to change to No as they are not part of the customer plan 

In [14]:
#delete 11 rows from 'total charges'

df = df.dropna(subset=['TotalCharges'])

The 11 missing values are less than 1% of the data, we can delete without altering the results

In [15]:
#Encode Data

df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

In [16]:
#Split the test set and use stratification because it is classification.

from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split( 
    X,
    y,
    test_size=0.2,
    random_state=12345,
    stratify=y
)

In [17]:
#Split validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.25,
    random_state=12345,
    stratify=y_train_full
)

Validation set and this is the new distribution of the data (100% Data):
* Train set is now 60%
* Test set 20%
* Validation set 20%




In [18]:
#import metrics

from sklearn.metrics import roc_auc_score, accuracy_score


ROC-AUC (mandatory) and Accuracy for further information

In [19]:
#Baseline Model - Logistic Regression

from sklearn.linear_model import LogisticRegression


lr_model = LogisticRegression(max_iter=1000, random_state=12345)
lr_model.fit(X_train, y_train)

lr_probs = lr_model.predict_proba(X_val)[:, 1]
lr_preds = lr_model.predict(X_val)


lr_auc = roc_auc_score(y_val, lr_probs)
lr_acc = accuracy_score(y_val, lr_preds)


print('Logistic Regression')
print('ROC-AUC:', lr_auc)
print('Accuracy:', lr_acc)


Logistic Regression
ROC-AUC: 0.8472208561326493
Accuracy: 0.814498933901919


Logistic Regression was used as a baseline model due to its simplicity and interpretability. On the validation set, the model achieved a ROC-AUC of 0.847 and an accuracy of 0.814, providing a reasonable reference point for comparing more complex models.

In [20]:
# Model #2 - Random Forest

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    random_state=12345
)
rf_model.fit(X_train, y_train)

rf_probs = rf_model.predict_proba(X_val)[:, 1]
rf_preds = rf_model.predict(X_val)

rf_auc = roc_auc_score(y_val, rf_probs)
rf_acc = accuracy_score(y_val, rf_preds)

print('Random Forest')
print('ROC-AUC:', rf_auc)
print('Accuracy:', rf_acc)

Random Forest
ROC-AUC: 0.8905529297875974
Accuracy: 0.8521677327647477


The Random Forest model improved performance by combining multiple decision trees and capturing nonlinear relationships between features. On the validation set, it achieved a ROC-AUC of 0.891 and an accuracy of 0.852, demonstrating stronger predictive capability than Logistic Regression.

In [21]:
#Model #3 - LightGBM

from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=30,
    random_state=12345
)
lgbm_model.fit(X_train, y_train)

lgbm_probs = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_preds = lgbm_model.predict(X_val)

lgbm_auc = roc_auc_score(y_val, lgbm_probs)
lgbm_acc = accuracy_score(y_val, lgbm_preds)

print('LightGBM')
print('ROC-AUC:', lgbm_auc)
print('Accuracy:', lgbm_acc)


LightGBM
ROC-AUC: 0.9358922923213112
Accuracy: 0.8997867803837953


The LightGBM model further improved performance by using gradient boosting to iteratively refine predictions and capture complex feature interactions. On the validation set, it achieved a ROC-AUC of 0.936 and an accuracy of 0.900, outperforming the previous models.

In [22]:
# Hyperparameter tuning
lgbmt_model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=60,
    random_state=12345
)

lgbmt_model.fit(X_train, y_train)


lgbmt_probs = lgbmt_model.predict_proba(X_val)[:, 1]
lgbmt_preds = lgbmt_model.predict(X_val)

lgbmt_auc = roc_auc_score(y_val, lgbmt_probs)
lgbmt_acc = accuracy_score(y_val, lgbmt_preds)


print('LightGBM Tunned')
print('ROC-AUC:', lgbmt_auc)
print('Accuracy:', lgbmt_acc)


LightGBM Tunned
ROC-AUC: 0.9381428889429572
Accuracy: 0.9019189765458422


After tuning the LightGBM hyperparameters, the model achieved a slightly higher ROC-AUC of 0.938 and an accuracy of 0.902 on the validation set. Although the improvement over the default LightGBM model was modest, the tuned version produced the best overall performance and was therefore selected as the final model.

In [23]:
#Visualize in a table all results

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'LightGBM','LightGBM Tunned'],
    'ROC-AUC': [lr_auc, rf_auc, lgbm_auc, lgbmt_auc],
    'Accuracy': [lr_acc, rf_acc, lgbm_acc, lgbmt_acc]
})

results.sort_values(by='ROC-AUC', ascending=False)

,Model,ROC-AUC,Accuracy
3,LightGBM Tunned,0.938143,0.901919
2,LightGBM,0.935892,0.899787
1,Random Forest,0.890553,0.852168
0,Logistic Regression,0.847221,0.814499


In [24]:
#Evaluate on test

final_probs = lgbmt_model.predict_proba(X_test)[:,1]
final_preds = lgbmt_model.predict(X_test)

final_auc = roc_auc_score(y_test, final_probs)
final_acc = accuracy_score(y_test, final_preds)

print('Final Model: LightGBM Tunned')
print('Test ROC-AUC:', final_auc)
print('Test Accuracy:', final_acc)

Final Model: LightGBM Tunned
Test ROC-AUC: 0.9348556460338249
Test Accuracy: 0.892679459843639


After selecting the tuned LightGBM model based on the validation results, the model was evaluated on the test set, which was kept separate during the training and tuning process. The final model achieved a ROC-AUC score of 0.935 and an accuracy of 0.893, confirming strong predictive performance and demonstrating that the model effectively distinguishes between customers who remain active and those who churn. These results exceed the required threshold for the project and indicate that the selected model generalizes well to unseen data.

# Conclusions

The objective of this project was to develop a model capable of predicting whether a customer remains active, using the condition EndDate == 'No' as the target variable. Several classification models were trained and evaluated, including Logistic Regression as a baseline model, Random Forest as a tree-based ensemble model, and LightGBM as a gradient boosting model. The dataset was divided into training, validation, and test sets using stratified sampling to preserve class distribution. The validation set was used to compare models and tune hyperparameters, while ROC-AUC was used as the primary evaluation metric and accuracy as a secondary metric.

Among the evaluated models, the tuned LightGBM model achieved the best performance on the validation set and was therefore selected as the final model. After fixing the hyperparameters, the final evaluation was performed on the test set, where the model achieved a ROC-AUC score of 0.935 and an accuracy of 0.893, outperforming Logistic Regression and Random Forest. These results exceed the required threshold of 0.85 ROC-AUC, demonstrating that the model effectively distinguishes between customers who remain active and those who churn while also generalizing well to unseen data.

### Business Interpretation

As requested, the developed model can help the company identify customers who are likely to stop using the service. By predicting customer retention in advance, the company can implement targeted retention strategies, such as personalized offers or improved customer support, to reduce churn and maintain long-term customer relationships.
